# Step 2.3.2 — YOLO Monocular Depth (NEW — no LiDAR) ✅

| | |
|---|---|
| **Input** | `output/step_2/yolo/<sample>/<camera>.json` (2D boxes, from Step 2.3) |
| | `output/step_1/camera_meta.json` (from Step 1.1) |
| **Outputs** | `output/step_2/yolo_mono/<sample>/<camera>.json` — each detection now has a monocular-only estimated 3D global position |
| | `output/step_2/yolo_mono_summary.csv` — depth distribution per class |
| **Used by** | Step 3.4 (camera-mono tracker), Step 5 (TTC estimation, true monocular baseline) |

---

### Why this notebook exists

Step 2.3.1 lifts each YOLO 2D box to 3D by borrowing depth from LiDAR — useful for tracking, but it means the "Camera" row in the evaluation table is not actually a camera-only result; it is camera detection + LiDAR depth. This notebook adds a genuinely LiDAR-free baseline: the classic **similar-triangles / apparent-size** monocular depth estimate that real single-camera ADAS systems without LiDAR/radar rely on.

### How it works

1. For each YOLO detection, take the box's pixel height (`ymax - ymin`).
2. Look up an assumed real-world height for that object class (`OBJECT_HEIGHT_M` below) — this is the one unavoidable assumption of monocular depth-from-height; there is no way to recover absolute scale from a single 2D image without *some* prior.
3. Depth estimate: `Z = (real_height_m * fy) / pixel_height`, where `fy` is the camera's vertical focal length in pixels (from `camera_intrinsic`) — this is the pinhole-camera similar-triangles relationship.
4. Un-project the box center pixel at that depth back into the camera's own 3D frame: `X = (u_center - cx) * Z / fx`, `Y = (v_center - cy) * Z / fy`.
5. Lift that camera-frame 3D point into the global frame with `point_to_global()` (the same transform-chain utility Step 1.1/2.3.1/3.1/3.2 already use), using the camera's own `ego_pose` + `calibrated_sensor` for this sample.

### Known limitation (flagging, not hiding)

`OBJECT_HEIGHT_M` is a fixed per-class average — a real object's actual height varies around it (a child vs. an adult, a hatchback vs. an SUV, a school bus vs. a minibus), and every such deviation becomes a directly proportional depth error. A box partially cut off by the image edge (truncation) or occluded by another object also silently shrinks the measured pixel height, which the formula reads as "further away" and overestimates depth. Real monocular-only perception stacks either accept this variance or spend a full 3D bounding-box regression network to reduce it — a plain apparent-size estimate does not; this notebook does not attempt to correct for either.

In [1]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Verify config.py exists
# ─────────────────────────────────────────────────────────────────

from pathlib import Path

if not Path("config.py").exists():
    raise FileNotFoundError("config.py not found. Copy it from the repo root.")

from config import STEP1_DIR, STEP2_DIR

YOLO_IN_DIR   = STEP2_DIR / "yolo"
MONO_OUT_DIR  = STEP2_DIR / "yolo_mono"
MONO_OUT_DIR.mkdir(parents=True, exist_ok=True)

cam_meta_path = STEP1_DIR / "camera_meta.json"

for p, name in [(YOLO_IN_DIR, "Step 2.3 YOLO output"), (cam_meta_path, "Step 1.1 camera_meta.json")]:
    if not p.exists():
        raise FileNotFoundError(f"{name} not found at {p} — run that step first.")

print(f"✅ YOLO_IN_DIR  : {YOLO_IN_DIR}")
print(f"✅ MONO_OUT_DIR : {MONO_OUT_DIR}")

config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output
✅ YOLO_IN_DIR  : F:\Sensor fusion Research\output\step_2\yolo
✅ MONO_OUT_DIR : F:\Sensor fusion Research\output\step_2\yolo_mono


In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Assumed real-world object heights (metres) + geometry import
#
# This is the one scale assumption similar-triangles depth cannot avoid:
# a single 2D image has no absolute scale without a size prior. Values
# are rough class averages, not measured per-object.
# ─────────────────────────────────────────────────────────────────

from src.geometry import point_to_global

OBJECT_HEIGHT_M = {
    "person": 1.7,
    "bicycle": 1.2,
    "car": 1.5,
    "motorcycle": 1.3,
    "bus": 3.4,
    "truck": 3.5,
}
DEFAULT_HEIGHT_M = 1.5   # fallback for any class not in the table above

print("✅ OBJECT_HEIGHT_M loaded:", OBJECT_HEIGHT_M)

✅ OBJECT_HEIGHT_M loaded: {'person': 1.7, 'bicycle': 1.2, 'car': 1.5, 'motorcycle': 1.3, 'bus': 3.4, 'truck': 3.5}


In [3]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Main pipeline: YOLO 2D box -> similar-triangles depth -> global 3D
# (No LiDAR anywhere in this cell — genuinely monocular-only.)
# ─────────────────────────────────────────────────────────────────

import json
from tqdm import tqdm

with open(cam_meta_path) as f:
    cam_meta_data = json.load(f)

summary_rows = []
n_total_detections = 0
n_degenerate_boxes = 0

for sample_id, cam_views in tqdm(cam_meta_data.items(), desc="Monocular depth"):
    yolo_sample_dir = YOLO_IN_DIR / sample_id
    if not yolo_sample_dir.exists():
        continue

    out_sample_dir = MONO_OUT_DIR / sample_id
    out_sample_dir.mkdir(parents=True, exist_ok=True)

    for cam_name, cam_entry in cam_views.items():
        yolo_file = yolo_sample_dir / f"{cam_name}.json"
        if not yolo_file.exists():
            continue

        with open(yolo_file) as f:
            detections = json.load(f)

        cam_ego_pose = cam_entry["ego_pose"]
        cam_calib = cam_entry["calibrated_sensor"]
        K = cam_calib["camera_intrinsic"]
        fx, fy, cx, cy = K[0][0], K[1][1], K[0][2], K[1][2]

        enriched_detections = []
        depths_this_cam = []

        for det in detections:
            n_total_detections += 1
            pixel_height = det["ymax"] - det["ymin"]

            enriched = dict(det)

            if pixel_height <= 0:
                n_degenerate_boxes += 1
                enriched["has_3d_position"] = False
                enriched_detections.append(enriched)
                continue

            real_height_m = OBJECT_HEIGHT_M.get(det["class_name"], DEFAULT_HEIGHT_M)
            depth = real_height_m * fy / pixel_height

            u_center = (det["xmin"] + det["xmax"]) / 2.0
            v_center = (det["ymin"] + det["ymax"]) / 2.0
            x_cam = (u_center - cx) * depth / fx
            y_cam = (v_center - cy) * depth / fy

            global_xyz = point_to_global([x_cam, y_cam, depth], cam_ego_pose, cam_calib)

            enriched["has_3d_position"] = True
            enriched["global_x"] = float(global_xyz[0])
            enriched["global_y"] = float(global_xyz[1])
            enriched["global_z"] = float(global_xyz[2])
            enriched["estimated_depth_m"] = float(depth)
            enriched["assumed_height_m"] = real_height_m
            enriched_detections.append(enriched)
            depths_this_cam.append(depth)

        with open(out_sample_dir / f"{cam_name}.json", "w") as f:
            json.dump(enriched_detections, f, indent=2)

        if depths_this_cam:
            summary_rows.append({
                "sample_id": sample_id, "camera": cam_name,
                "num_detections": len(detections),
                "mean_depth_m": sum(depths_this_cam) / len(depths_this_cam),
            })

print(f"\n✅ Step 2.3.2 complete.")
print(f"   Total detections           : {n_total_detections}")
print(f"   Degenerate boxes (skipped) : {n_degenerate_boxes}")
print(f"   Saved to: {MONO_OUT_DIR}")

Monocular depth:   0%|          | 0/404 [00:00<?, ?it/s]

Monocular depth:   0%|          | 1/404 [00:01<10:08,  1.51s/it]

Monocular depth:   0%|          | 2/404 [00:01<04:40,  1.43it/s]

Monocular depth:   1%|          | 4/404 [00:01<01:57,  3.41it/s]

Monocular depth:   2%|▏         | 7/404 [00:01<00:58,  6.83it/s]

...

Monocular depth:  98%|█████████▊| 395/404 [00:37<00:00, 25.63it/s]

Monocular depth:  99%|█████████▉| 399/404 [00:38<00:00, 28.29it/s]

Monocular depth: 100%|█████████▉| 402/404 [00:38<00:00, 27.58it/s]

Monocular depth: 100%|██████████| 404/404 [00:38<00:00, 10.56it/s]


✅ Step 2.3.2 complete.
   Total detections           : 6045
   Degenerate boxes (skipped) : 0
   Saved to: F:\Sensor fusion Research\output\step_2\yolo_mono


In [4]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 — Summary: depth distribution by class
# ─────────────────────────────────────────────────────────────────

import pandas as pd

summary_df = pd.DataFrame(summary_rows)
summary_path = STEP2_DIR / "yolo_mono_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"✅ Summary saved: {summary_path}")

by_class_depths = {}
for sample_dir in MONO_OUT_DIR.iterdir():
    if not sample_dir.is_dir():
        continue
    for cam_file in sample_dir.glob("*.json"):
        with open(cam_file) as f:
            dets = json.load(f)
        for det in dets:
            if det.get("has_3d_position"):
                by_class_depths.setdefault(det["class_name"], []).append(det["estimated_depth_m"])

print("\nMean estimated depth by class:")
for cls, depths in sorted(by_class_depths.items()):
    print(f"  {cls:<12} n={len(depths):<6} mean_depth={sum(depths)/len(depths):.1f}m")

✅ Summary saved: F:\Sensor fusion Research\output\step_2\yolo_mono_summary.csv



Mean estimated depth by class:
  bicycle      n=56     mean_depth=12.6m
  bus          n=184    mean_depth=18.4m
  car          n=3939   mean_depth=20.2m
  motorcycle   n=21     mean_depth=10.0m
  person       n=1446   mean_depth=14.2m
  truck        n=399    mean_depth=16.6m
